<a href="https://colab.research.google.com/github/DivyAgrawal11/Automated-Equity-Valuation-Engine/blob/main/Stock_Valuation_Dashboard_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# INSTALL DEPENDENCIES
!apt-get update -qq
!apt-get install -y wkhtmltopdf
!pip install -q yfinance plotly pdfkit jinja2 kaleido==0.2.1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
wkhtmltopdf is already the newest version (0.12.6-2).
0 upgraded, 0 newly installed, 0 to remove and 46 not upgraded.


In [65]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import pdfkit
from jinja2 import Template
import base64
import warnings

# Suppress pandas FutureWarnings from yfinance
warnings.simplefilter(action='ignore', category=FutureWarning)

# Configuration for pdfkit to find the installed binary in Colab
path_wkhtmltopdf = '/usr/bin/wkhtmltopdf'
config = pdfkit.configuration(wkhtmltopdf=path_wkhtmltopdf)

print("Dalal Street Valuation Engine configured successfully.")

Dalal Street Valuation Engine configured successfully.


In [66]:
class IndianFinancialDataEngineV2:
    def __init__(self, ticker_symbol: str):
        if not ticker_symbol.endswith('.NS') and not ticker_symbol.endswith('.BO'):
            ticker_symbol += '.NS'
        self.ticker_symbol = ticker_symbol
        self.ticker = yf.Ticker(ticker_symbol)
        self.info = self.ticker.info

    def get_market_stats(self):
        """Pulls Bloomberg-style statistics."""
        return {
            "52w_high": self.info.get('fiftyTwoWeekHigh', 0),
            "52w_low": self.info.get('fiftyTwoWeekLow', 0),
            "beta": self.info.get('beta', 1.0),
            "div_yield": self.info.get('dividendYield', 0),
            "shares_out": self.info.get('sharesOutstanding', 0)
        }

    def fetch_historical_financials(self) -> pd.DataFrame:
        try:
            inc = self.ticker.financials.T
            bs = self.ticker.balance_sheet.T
            cf = self.ticker.cashflow.T
            df = inc.join(bs, how='outer').join(cf, how='outer', rsuffix='_cf')
            df.sort_index(ascending=True, inplace=True)
            return df
        except Exception as e:
            raise ValueError(f"Failed to fetch data for {self.ticker_symbol}: {e}")

    def calculate_margin_trends(self, df: pd.DataFrame):
        """Calculates 5-year margin trends."""
        # Ensure we have data
        margins = pd.DataFrame()
        margins['Operating Margin'] = df.get('EBIT', 0) / df.get('Total Revenue', 1)
        margins['Net Margin'] = df.get('Net Income', 0) / df.get('Total Revenue', 1)
        return margins.tail(5) # Last 5 years

    def safe_cagr(self, beginning_value, ending_value, periods=3):
        """Calculates CAGR safely, handling negative base years."""
        if beginning_value <= 0 or periods <= 0: return 0
        return ((ending_value / beginning_value) ** (1 / periods)) - 1

    def extract_core_metrics(self, df: pd.DataFrame) -> dict:
        latest = df.iloc[-1]

        # Determine 3-Year Historicals (yfinance usually provides 4 years of annual data)
        try:
            past_3y = df.iloc[-4] if len(df) >= 4 else df.iloc[0]
            years_available = len(df) - 1 if len(df) > 1 else 1
        except:
            past_3y = latest
            years_available = 1

        current_price = self.info.get('currentPrice', self.info.get('regularMarketPrice', 0))
        shares_out = self.info.get('sharesOutstanding', 0)
        tax_rate = self.info.get('taxRate', 0.2517)

        # Absolute figures for Math
        market_cap_abs = current_price * shares_out
        total_debt_abs = self.info.get('totalDebt', latest.get('Total Debt', 0))
        cash_abs = self.info.get('totalCash', latest.get('Cash And Cash Equivalents', 0))

        # Income & Cash Flow Items
        revenue_abs = latest.get('Total Revenue', 0)
        ebit_abs = latest.get('EBIT', 0)
        net_income = latest.get('Net Income', 0)
        ebitda = latest.get('EBITDA', ebit_abs) # Fallback to EBIT if missing
        operating_cf = latest.get('Operating Cash Flow', 0)
        capex = latest.get('Capital Expenditure', 0) # Usually negative in yfinance
        fcf = operating_cf + capex

        # Balance Sheet Items for Quality Scorecard
        total_equity = latest.get('Stockholders Equity', latest.get('Total Equity Gross Minority Interest', 1))
        invested_capital = total_debt_abs + total_equity - cash_abs

        to_crores = 10_000_000

        return {
            "current_price": current_price,
            "shares_out": shares_out,
            "market_cap_abs": market_cap_abs,
            "total_debt_abs": total_debt_abs,
            "cash_abs": cash_abs,
            "revenue_abs": revenue_abs,
            "ebit_abs": ebit_abs,
            "tax_rate": tax_rate,
            "beta": self.info.get('beta', 1.0),

            # --- V2 Formatting ---
            "market_cap_cr": market_cap_abs / to_crores,
            "total_debt_cr": total_debt_abs / to_crores,

            # --- V2 Quality Scorecard ---
            "roe": net_income / total_equity if total_equity > 0 else 0,
            "roic": (ebit_abs * (1 - tax_rate)) / invested_capital if invested_capital > 0 else 0,
            "ebitda_margin": ebitda / revenue_abs if revenue_abs > 0 else 0,
            "fcf_margin": fcf / revenue_abs if revenue_abs > 0 else 0,
            "debt_to_equity": total_debt_abs / total_equity if total_equity > 0 else 0,

            # --- V2 Historical Growth (CAGR) ---
            "rev_cagr": self.safe_cagr(past_3y.get('Total Revenue', 0), revenue_abs, years_available),
            "ebitda_cagr": self.safe_cagr(past_3y.get('EBITDA', past_3y.get('EBIT', 0)), ebitda, years_available),
            "ni_cagr": self.safe_cagr(past_3y.get('Net Income', 0), net_income, years_available)
        }

In [67]:
class IndianDCFModelerV2:
    def __init__(self, metrics: dict, india_10y_gsec: float = 0.0684, india_erp: float = 0.0700):
        self.metrics = metrics
        self.rf = india_10y_gsec
        self.erp = india_erp

    def calculate_wacc(self, cost_of_debt: float = 0.085) -> float:
        E = self.metrics['market_cap_abs']
        D = self.metrics['total_debt_abs']
        V = E + D
        if V == 0: return 0.12
        Re = self.rf + self.metrics['beta'] * self.erp
        Rd = cost_of_debt * (1 - self.metrics['tax_rate'])
        return (E/V * Re) + (D/V * Rd)

    def project_fcff(self, years: int = 5, rev_growth: float = 0.10, ebit_margin: float = 0.18):
        projections = []
        current_rev = self.metrics['revenue_abs']
        for year in range(1, years + 1):
            rev = current_rev * ((1 + rev_growth) ** year)
            ebit = rev * ebit_margin
            nopat = ebit * (1 - self.metrics['tax_rate'])
            projections.append(nopat) # Simplified FCFF proxy
        return projections

    def compute_intrinsic_value(self, fcff_projections: list, wacc: float, terminal_growth: float = 0.045) -> dict:
        """V2: Returns a full breakdown dictionary instead of just a float."""
        if terminal_growth >= wacc:
            return {"share_price": np.nan, "pv_fcff": 0, "pv_tv": 0, "ev": 0, "eq": 0}

        pv_fcff = sum([cf / ((1 + wacc) ** (i + 1)) for i, cf in enumerate(fcff_projections)])

        final_year_cf = fcff_projections[-1]
        terminal_value = (final_year_cf * (1 + terminal_growth)) / (wacc - terminal_growth)
        pv_tv = terminal_value / ((1 + wacc) ** len(fcff_projections))

        enterprise_value = pv_fcff + pv_tv
        tv_contribution = (pv_tv / enterprise_value) * 100

        equity_value = enterprise_value - self.metrics['total_debt_abs'] + self.metrics['cash_abs']
        share_price = equity_value / self.metrics['shares_out'] if self.metrics['shares_out'] > 0 else 0

        return {
            "pv_fcff": pv_fcff,
            "pv_tv": pv_tv,
            "enterprise_value": enterprise_value,
            "equity_value": equity_value,
            "share_price": share_price,
            "tv_cont": tv_contribution
        }

    def get_investment_verdict(self, current_price, target_price):
        if np.isnan(target_price) or target_price <= 0:
            return "N/A", np.nan
        upside = (target_price / current_price) - 1
        if upside > 0.40: return "STRONG BUY", upside
        if upside > 0.15: return "BUY", upside
        if upside > -0.10: return "HOLD", upside
        return "SELL", upside

In [74]:
class VisualizationEngineV2:

    @staticmethod
    def margin_chart(margins_df):
        if margins_df.empty:
            return ""

        fig = go.Figure()
        for col in margins_df.columns:
            fig.add_trace(go.Scatter(x=margins_df.index, y=margins_df[col], mode='lines+markers', name=col))

        fig.update_layout(
            title="5Y Margin Trends", height=300, width=500, plot_bgcolor='white',
            yaxis_tickformat='.1%', legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )
        return base64.b64encode(fig.to_image(format="png")).decode("utf-8")

    @staticmethod
    def sensitivity_heatmap(model: IndianDCFModelerV2, wacc_base: float, g_base: float) -> str:
        wacc_range = np.linspace(wacc_base - 0.02, wacc_base + 0.02, 5)
        g_range = np.linspace(g_base - 0.015, g_base + 0.015, 5)

        prices = np.zeros((len(wacc_range), len(g_range)))
        text_labels = []

        for i, w in enumerate(wacc_range):
            row_text = [] # <-- This resets for every row
            for j, g in enumerate(g_range):
                projections = model.project_fcff()
                result = model.compute_intrinsic_value(projections, w, g)

                val = result['share_price']
                prices[i, j] = val

                # Check the defined variable and append to the row
                if np.isnan(val) or val <= 0:
                    row_text.append("N/A")
                else:
                    row_text.append(f"₹{val:.1f}")

            # Append the completed row to the main labels array
            text_labels.append(row_text)

        fig = go.Figure(data=go.Heatmap(
            z=prices, x=[f"{g*100:.1f}%" for g in g_range], y=[f"{w*100:.1f}%" for w in wacc_range],
            colorscale='RdYlGn', text=text_labels, texttemplate="%{text}", hoverinfo="text"
        ))

        fig.update_layout(
            title="Implied Share Price Sensitivity", xaxis_title="Terminal Growth Rate (g)",
            yaxis_title="WACC", width=600, height=300, plot_bgcolor='white'
        )

        return base64.b64encode(fig.to_image(format="png")).decode("utf-8")

    @staticmethod
    def valuation_waterfall(dcf_breakdown: dict, total_debt: float, total_cash: float) -> str:
        to_cr = 10_000_000
        fig = go.Figure(go.Waterfall(
            name="Valuation Bridge", orientation="v", measure=["absolute", "relative", "relative", "total"],
            x=["Enterprise Value", "Less: Total Debt", "Add: Cash", "Equity Value"],
            textposition="outside",
            text=[
                f"₹{dcf_breakdown['enterprise_value']/to_cr:,.0f}",
                f"-₹{total_debt/to_cr:,.0f}",
                f"+₹{total_cash/to_cr:,.0f}",
                f"₹{dcf_breakdown['equity_value']/to_cr:,.0f}"
            ],
            y=[
                dcf_breakdown['enterprise_value']/to_cr,
                -total_debt/to_cr,
                total_cash/to_cr,
                dcf_breakdown['equity_value']/to_cr
            ],
            connector={"line": {"color": "rgb(63, 63, 63)"}},
            decreasing={"marker": {"color": "#ef4444"}},
            increasing={"marker": {"color": "#10b981"}},
            totals={"marker": {"color": "#3b82f6"}}
        ))

        fig.update_layout(
            title="Valuation Waterfall (₹ Crores)", width=600, height=600,
            plot_bgcolor='white', showlegend=False
        )

        return base64.b64encode(fig.to_image(format="png")).decode("utf-8")

In [75]:
html_template_v3 = """
<!DOCTYPE html>
<html>
<head>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; color: #1e293b; margin: 0; font-size: 13px; line-height: 1.5; background-color: #fff; }
        .report-header { text-align: center; margin-bottom: 30px; border-bottom: 2px solid #1e3a8a; padding-bottom: 15px; }
        .report-header h1 { color: #1e3a8a; margin: 0; font-size: 28px; text-transform: uppercase; letter-spacing: 1px; }
        h3 { color: #2563eb; margin-bottom: 12px; border-bottom: 2px solid #e2e8f0; padding-bottom: 6px; font-size: 15px; }
        .flex-container { display: flex; gap: 25px; margin-bottom: 25px; }
        .box { background: #f8fafc; padding: 20px; border-radius: 8px; border-top: 5px solid #2563eb; flex: 1; box-shadow: 0 4px 6px rgba(0,0,0,0.05); text-align: center; }
        .table-container { flex: 1; background: #ffffff; padding: 15px; border-radius: 8px; border: 1px solid #cbd5e1; box-shadow: 0 2px 4px rgba(0,0,0,0.02); }
        .stats-table { width: 100%; border-collapse: collapse; }
        .stats-table th { background-color: #f1f5f9; color: #475569; padding: 10px 12px; text-align: left; font-weight: 600; border-bottom: 2px solid #cbd5e1; }
        .stats-table td { padding: 10px 12px; border-bottom: 1px solid #e2e8f0; }
        .stats-table tr:last-child td { border-bottom: none; }
        .highlight { background-color: #eff6ff; font-weight: bold; color: #1e3a8a; }
        .page-break { page-break-before: always; }
        .chart-container { text-align: center; padding: 15px; border: 1px solid #e2e8f0; border-radius: 8px; background: #fff; margin-bottom: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.02); }
        img { max-width: 100%; height: auto; display: block; margin: 0 auto; }
    </style>
</head>
<body>
    <div class="report-header">
        <h1>Equity Valuation Report: {{ ticker }}</h1>
    </div>

    <div class="flex-container">
        <div class="box" style="flex: 0.8; display: flex; flex-direction: column; justify-content: center;">
            <h3 style="border:none; margin-bottom: 5px; color:#475569;">Investment Verdict</h3>
            <h1 style="color:{{ 'green' if verdict in ['BUY', 'STRONG BUY'] else ('red' if verdict == 'SELL' else '#64748b') }}; font-size: 32px; margin: 10px 0;">{{ verdict }}</h1>
            <p style="margin: 4px 0; font-size: 16px;"><strong>Target Price:</strong>
                {% if isnan(base_dcf.share_price) %}N/A{% else %}₹{{ "%.2f"|format(base_dcf.share_price) }}{% endif %}
            </p>
            <p style="margin: 4px 0; font-size: 16px; color: {{ 'green' if upside > 0 else 'red' }};"><strong>Upside Potential:</strong>
                {% if isnan(upside) %}N/A{% else %}{{ "%.1f"|format(upside*100) }}%{% endif %}
            </p>
        </div>

        <div class="table-container">
            <h3>Market Snapshot</h3>
            <table class="stats-table">
                <tr><td>Current Market Price (CMP)</td><td style="font-weight: bold;">₹{{ "%.2f"|format(metrics.current_price) }}</td></tr>
                <tr><td>Market Capitalization</td><td>₹{{ "{:,.0f}".format(metrics.market_cap_cr) }} Cr</td></tr>
                <tr><td>52W High / Low</td><td>₹{{ "%.2f"|format(stats['52w_high']) }} / ₹{{ "%.2f"|format(stats['52w_low']) }}</td></tr>
                <tr><td>Beta</td><td>{{ "%.2f"|format(stats.beta) }}</td></tr>
                <tr><td>Dividend Yield</td><td>{{ "%.2f"|format(stats.div_yield*100) }}%</td></tr>
            </table>
        </div>
    </div>

    <div class="flex-container">
        <div class="table-container">
            <h3>Quality Scorecard</h3>
            <table class="stats-table">
                <tr><td>Return on Equity (ROE)</td><td>{{ "%.2f"|format(metrics.roe * 100) }}%</td></tr>
                <tr><td>Return on Invested Capital (ROIC)</td><td>{{ "%.2f"|format(metrics.roic * 100) }}%</td></tr>
                <tr><td>EBITDA Margin</td><td>{{ "%.2f"|format(metrics.ebitda_margin * 100) }}%</td></tr>
                <tr><td>FCF Margin</td><td>{{ "%.2f"|format(metrics.fcf_margin * 100) }}%</td></tr>
                <tr><td>Debt to Equity</td><td>{{ "%.2f"|format(metrics.debt_to_equity) }}x</td></tr>
            </table>
        </div>

        <div class="table-container">
            <h3>Historical Growth (3Y CAGR)</h3>
            <table class="stats-table">
                <tr><td>Revenue CAGR</td><td>{{ "%.2f"|format(metrics.rev_cagr * 100) }}%</td></tr>
                <tr><td>EBITDA CAGR</td><td>{{ "%.2f"|format(metrics.ebitda_cagr * 100) }}%</td></tr>
                <tr><td>Net Income CAGR</td><td>{{ "%.2f"|format(metrics.ni_cagr * 100) }}%</td></tr>
            </table>
        </div>
    </div>

    <div class="flex-container">
        <div class="table-container" style="flex: 1.2;">
            <h3>Intrinsic Valuation Breakdown</h3>
            <table class="stats-table">
                <tr><th>Component</th><th>Value (₹ Cr)</th></tr>
                <tr><td>PV of Explicit Forecast</td><td>₹{{ "{:,.2f}".format(base_dcf.pv_fcff / 10000000) }}</td></tr>
                <tr><td>PV of Terminal Value ({{ "%.1f"|format(base_dcf.tv_cont) }}%)</td><td>₹{{ "{:,.2f}".format(base_dcf.pv_tv / 10000000) }}</td></tr>
                <tr class="highlight"><td>Enterprise Value (EV)</td><td>₹{{ "{:,.2f}".format(base_dcf.enterprise_value / 10000000) }}</td></tr>
                <tr><td>Add: Cash & Equivalents</td><td>+ ₹{{ "{:,.2f}".format(metrics.cash_abs / 10000000) }}</td></tr>
                <tr><td>Less: Total Debt</td><td>- ₹{{ "{:,.2f}".format(metrics.total_debt_abs / 10000000) }}</td></tr>
                <tr class="highlight"><td>Implied Equity Value</td><td>₹{{ "{:,.2f}".format(base_dcf.equity_value / 10000000) }}</td></tr>
            </table>
        </div>

        <div class="table-container" style="flex: 0.8;">
            <h3>Valuation Scenarios</h3>
            <table class="stats-table">
                <tr style="background-color: #fef2f2; color: #991b1b;">
                    <td><strong>Bear Case Price</strong></td>
                    <td>{% if isnan(bear_dcf.share_price) %}N/A{% else %}₹{{ "%.2f"|format(bear_dcf.share_price) }}{% endif %}</td>
                </tr>
                <tr class="highlight">
                    <td><strong>Base Case Price</strong></td>
                    <td>{% if isnan(base_dcf.share_price) %}N/A{% else %}₹{{ "%.2f"|format(base_dcf.share_price) }}{% endif %}</td>
                </tr>
                <tr style="background-color: #ecfdf5; color: #065f46;">
                    <td><strong>Bull Case Price</strong></td>
                    <td>{% if isnan(bull_dcf.share_price) %}N/A{% else %}₹{{ "%.2f"|format(bull_dcf.share_price) }}{% endif %}</td>
                </tr>
            </table>
        </div>
    </div>

    <div class="page-break"></div>
    <div class="report-header" style="margin-top: 20px;">
        <h1>Visualization Appendix</h1>
    </div>
    <div class="flex-container">
        <div class="chart-container" style="flex: 1;"><img src="data:image/png;base64,{{ waterfall_img }}" /></div>
        <div class="chart-container" style="flex: 1;"><img src="data:image/png;base64,{{ margin_chart_img }}" /></div>
    </div>
    <div class="chart-container">
        <img src="data:image/png;base64,{{ heatmap_img }}" style="max-width: 80%;" />
    </div>
</body>
</html>
"""

def generate_pdf_report_v3(data_dict, output_filename):
    template = Template(html_template_v3)
    html_out = template.render(**data_dict)

    options = {
        'page-size': 'A4',
        'margin-top': '0.6in',
        'margin-right': '0.6in',
        'margin-bottom': '0.6in',
        'margin-left': '0.6in',
        'encoding': "UTF-8",
        'enable-local-file-access': None
    }

    pdfkit.from_string(html_out, output_filename, options=options, configuration=config)
    print(f"Report generated: {output_filename}")

In [76]:
def run_v2_pipeline(ticker_symbol: str, assumptions: dict):
    print(f"Initiating V2 Dalal Street pipeline for {ticker_symbol}...")

    # 1. Fetch & Extract
    engine = IndianFinancialDataEngineV2(ticker_symbol)
    stats = engine.get_market_stats()
    df = engine.fetch_historical_financials()
    metrics = engine.extract_core_metrics(df)

    # 2. Build Model
    model = IndianDCFModelerV2(metrics)
    wacc = model.calculate_wacc()

    base_rev = assumptions.get('base_rev_growth', 0.10)
    base_margin = assumptions.get('base_ebit_margin', 0.18)
    term_growth = assumptions.get('terminal_growth', 0.045)

    # 3. Scenario Analysis
    cf_base = model.project_fcff(rev_growth=base_rev, ebit_margin=base_margin)
    cf_bear = model.project_fcff(rev_growth=base_rev - 0.03, ebit_margin=base_margin - 0.03)
    cf_bull = model.project_fcff(rev_growth=base_rev + 0.03, ebit_margin=base_margin + 0.03)

    base_dcf = model.compute_intrinsic_value(cf_base, wacc, terminal_growth=term_growth)
    bear_dcf = model.compute_intrinsic_value(cf_bear, wacc + 0.015, terminal_growth=term_growth - 0.01)
    bull_dcf = model.compute_intrinsic_value(cf_bull, wacc - 0.015, terminal_growth=term_growth + 0.01)

    verdict, upside = model.get_investment_verdict(metrics['current_price'], base_dcf['share_price'])

    # 4. Visualizations
    heatmap_img = VisualizationEngineV2.sensitivity_heatmap(model, wacc, term_growth)
    waterfall_img = VisualizationEngineV2.valuation_waterfall(
        base_dcf, metrics['total_debt_abs'], metrics['cash_abs']
    )
    margin_chart_img = VisualizationEngineV2.margin_chart(engine.calculate_margin_trends(df))

    # 5. Compile PDF
    report_data = {
        "ticker": engine.ticker_symbol,
        "metrics": metrics,
        "stats": stats,
        "base_dcf": base_dcf,
        "bear_dcf": bear_dcf,
        "bull_dcf": bull_dcf,
        "verdict": verdict,
        "upside": upside,
        "waterfall_img": waterfall_img,
        "heatmap_img": heatmap_img,
        "margin_chart_img": margin_chart_img,
        "isnan": np.isnan # Passing the numpy function to Jinja2 to check for invalid floats
    }

    clean_ticker = engine.ticker_symbol.replace('.NS', '').replace('.BO', '')
    output_file = f"{clean_ticker}_Valuation_Dashboard.pdf"

    generate_pdf_report_v3(report_data, output_file)

# --- EXECUTION BLOCK ---
try:
    assumptions = {
        'base_rev_growth': 0.08,
        'base_ebit_margin': 0.34,
        'terminal_growth': 0.045
    }
    run_v2_pipeline("ITC.NS", assumptions)

except Exception as e:
    print(f"V2 Pipeline failed: {e}")

Initiating V2 Dalal Street pipeline for ITC.NS...
Report generated: ITC_V2_Initiation_Report.pdf
